# Data processing - Players

In [1]:
import pandas as pd

DATA_PATH = "../../data/"

## Load data

In [2]:
def get_season(date):
    year = date.year
    if date.month >= 9:
        return f"{year}-{year+1}"
    else:
        return f"{year-1}-{year}"

In [3]:
player_stats = pd.read_csv(DATA_PATH + "/nba_database/PlayerStatistics.csv", dtype={10: str, 11: str, 15: str})

# filter to regular season and playoffs only
player_stats = player_stats[player_stats.gameType.isin(["Regular Season", "Playoffs"])]

# convert gameDateTimeEst to datetime and extract season
player_stats["gameDateTimeEst"] = pd.to_datetime(player_stats["gameDateTimeEst"])
player_stats["numMinutes"] = pd.to_numeric(player_stats["numMinutes"], errors="coerce")
player_stats = player_stats[player_stats["numMinutes"] > 0]

player_stats["season"] = player_stats["gameDateTimeEst"].apply(get_season)

# keep only data from 1980
player_stats = player_stats[player_stats["gameDateTimeEst"] >= pd.Timestamp("1980-10-01")]

# rename reboundsTotal to rebounds in the original dataframe for consistency
player_stats = player_stats.rename(columns={"reboundsTotal": "rebounds"}) 

display(player_stats)

,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,...,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,rebounds,foulsPersonal,turnovers,plusMinusPoints,season
1,Luke,Kennard,1628379.0,22500960,2026-03-12 22:30:00,Los Angeles,Lakers,Chicago,Bulls,Regular Season,...,0.0,0.0,0.000,1.0,0.0,1.0,2.0,0.0,-13.0,2025-2026
3,Jarred,Vanderbilt,1629020.0,22500960,2026-03-12 22:30:00,Los Angeles,Lakers,Chicago,Bulls,Regular Season,...,0.0,0.0,0.000,1.0,1.0,2.0,1.0,0.0,-3.0,2025-2026
4,Deandre,Ayton,1629028.0,22500960,2026-03-12 22:30:00,Los Angeles,Lakers,Chicago,Bulls,Regular Season,...,6.0,3.0,0.500,4.0,6.0,10.0,2.0,0.0,9.0,2025-2026
5,Luka,Doncic,1629029.0,22500960,2026-03-12 22:30:00,Los Angeles,Lakers,Chicago,Bulls,Regular Season,...,9.0,8.0,0.889,10.0,0.0,10.0,3.0,1.0,15.0,2025-2026
6,Rui,Hachimura,1629060.0,22500960,2026-03-12 22:30:00,Los Angeles,Lakers,Chicago,Bulls,Regular Season,...,0.0,0.0,0.000,0.0,1.0,1.0,1.0,0.0,16.0,2025-2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1350289,Louis,Orr,77770.0,28000009,1980-10-10 20:00:00,Indiana,Pacers,New Jersey,Nets,Regular Season,...,2.0,2.0,1.000,0.0,0.0,2.0,1.0,0.0,0.0,1980-1981
1350290,Cliff T.,Robinson,77986.0,28000009,1980-10-10 20:00:00,New Jersey,Nets,Indiana,Pacers,Regular Season,...,2.0,1.0,0.500,0.0,0.0,9.0,3.0,6.0,0.0,1980-1981
1350291,Jerry,Sichting,78146.0,28000009,1980-10-10 20:00:00,Indiana,Pacers,New Jersey,Nets,Regular Season,...,0.0,0.0,0.000,0.0,0.0,0.0,1.0,0.0,0.0,1980-1981
1350292,Jan,Van Breda Kolff,78400.0,28000009,1980-10-10 20:00:00,New Jersey,Nets,Indiana,Pacers,Regular Season,...,0.0,0.0,0.000,0.0,0.0,4.0,3.0,0.0,0.0,1980-1981


In [4]:
# add salary data
salaries = pd.read_csv(DATA_PATH + "/nba_salary/player_salaries.csv")
display(salaries)

,player,salary_usd,season,personId,firstName,lastName
0,Patrick Ewing,4250000,1990-1991,201607.0,Patrick,Ewing
1,Hot Rod Williams,3785000,1990-1991,73.0,Hot,Rod Williams
2,Hakeem Olajuwon,3175000,1990-1991,165.0,Hakeem,Olajuwon
3,Charles Barkley,2900000,1990-1991,787.0,Charles,Barkley
4,Chris Mullin,2850000,1990-1991,904.0,Chris,Mullin
...,...,...,...,...,...,...
18305,Gabe McGlothan,73153,2025-2026,1642440.0,Gabe,McGlothan
18306,Jaden Springer,70732,2025-2026,1630531.0,Jaden,Springer
18307,RayJ Dennis,67267,2025-2026,1642484.0,RayJ,Dennis
18308,Malik Williams,43870,2025-2026,1642013.0,Malik,Williams


### Per game

### Per season

In [5]:
# per season stats:
# Points
# Rebounds
# Assists
# Steal 
# Block
# Turnover
# Shot accuracy percentage (3p%, 2p%, FT%) total
# Proportion of 3pts % 
# Number of shot attempted
# Number of shot made
# Fouls
# +/-
# Salary
# Game played                
# Minutes played

columns_to_keep = ["season", "gameType", "personId", "firstName", "lastName", "points", 
                    "assists", "rebounds", "blocks", "steals", "turnovers", "fieldGoalsAttempted",
                    "fieldGoalsMade", "threePointersAttempted", "threePointersMade", "freeThrowsAttempted", 
                    "freeThrowsMade", "foulsPersonal", "plusMinusPoints", "numMinutes", "gameId", "win"]

agg_rules = {
    "points": "mean",
    "assists": "mean",
    "rebounds": "mean",
    "blocks": "mean",
    "steals": "mean",
    "turnovers": "mean",
    "fieldGoalsMade": "sum",
    "fieldGoalsAttempted": "sum",
    "threePointersMade": "sum",
    "threePointersAttempted": "sum",
    "freeThrowsMade": "sum",
    "freeThrowsAttempted": "sum",
    "plusMinusPoints": "mean",
    "foulsPersonal": "mean",
    "numMinutes": "mean",
    "win": "sum",
    "gameId": "count"
}

group_cols = ["season", "personId", "firstName", "lastName", "gameType"]
grouped = player_stats[columns_to_keep].groupby(group_cols)

# season agregated stats per team
player_stats_season = grouped.agg(agg_rules)
player_stats_season = player_stats_season.rename(columns={"gameId": "gamesPlayed"})

# add season totals for main box-score stats
main_stats = ["points", "assists", "rebounds", "blocks", "steals", "turnovers"]
main_totals = grouped[main_stats].sum().rename(columns={col: f"{col}Total" for col in main_stats})
player_stats_season = player_stats_season.join(main_totals)

# use minimum game thresholds for regular season / playoffs
regular_season_min_games = 15
playoffs_min_games = 4

game_type_idx = player_stats_season.index.get_level_values("gameType")
player_stats_season = player_stats_season[
    ((game_type_idx == "Regular Season") & (player_stats_season["gamesPlayed"] >= regular_season_min_games)) |
    ((game_type_idx == "Playoffs") & (player_stats_season["gamesPlayed"] >= playoffs_min_games))
]

# calculate shooting percentages on season totals
player_stats_season["proportionThreePoint"] = player_stats_season["threePointersAttempted"] / player_stats_season["fieldGoalsAttempted"]
player_stats_season["fieldGoalsPercentage"] = player_stats_season["fieldGoalsMade"] / player_stats_season["fieldGoalsAttempted"]
player_stats_season["threePointersPercentage"] = player_stats_season["threePointersMade"] / player_stats_season["threePointersAttempted"]
player_stats_season["freeThrowsPercentage"] = player_stats_season["freeThrowsMade"] / player_stats_season["freeThrowsAttempted"]

player_stats_season_flat = player_stats_season.reset_index().copy()

# join salaries on personId and season
salaries = salaries[["personId", "season", "salary_usd"]].rename(columns={"salary_usd": "salary"})
player_stats_season_flat = player_stats_season_flat.merge(
    salaries,
    left_on=["personId", "season"],
    right_on=["personId", "season"],
    how="left"
)

display(player_stats_season_flat)

,season,personId,firstName,lastName,gameType,points,assists,rebounds,blocks,steals,...,assistsTotal,reboundsTotal,blocksTotal,stealsTotal,turnoversTotal,proportionThreePoint,fieldGoalsPercentage,threePointersPercentage,freeThrowsPercentage,salary
0,1980-1981,229.0,James,Edwards,Regular Season,15.629630,2.617284,7.049383,1.382716,0.098765,...,212.0,571.0,112.0,8.0,82.0,0.003036,0.517206,0.000000,0.703170,NaN
1,1980-1981,305.0,Robert,Parish,Playoffs,15.000000,1.117647,8.588235,2.117647,0.411765,...,19.0,146.0,36.0,7.0,19.0,0.000000,0.493151,NaN,0.672414,NaN
2,1980-1981,305.0,Robert,Parish,Regular Season,18.926829,1.768293,9.475610,1.939024,0.402439,...,145.0,777.0,159.0,33.0,70.0,0.000858,0.544597,0.000000,0.710327,NaN
3,1980-1981,328.0,Rick,Mahorn,Regular Season,4.788462,0.480769,4.134615,0.480769,0.057692,...,25.0,215.0,25.0,3.0,13.0,0.000000,0.506849,NaN,0.692308,NaN
4,1980-1981,940.0,Earl,Cureton,Playoffs,1.333333,0.222222,1.000000,0.222222,0.111111,...,2.0,9.0,2.0,1.0,0.0,0.000000,0.333333,NaN,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25020,2025-2026,1642959.0,Chris,Youngblood,Regular Season,2.031250,0.343750,0.875000,0.093750,0.125000,...,11.0,28.0,3.0,4.0,2.0,0.850000,0.333333,0.313725,0.900000,636435.0
25021,2025-2026,1642959.0,Chris,Youngblood,Regular Season,2.031250,0.343750,0.875000,0.093750,0.125000,...,11.0,28.0,3.0,4.0,2.0,0.850000,0.333333,0.313725,0.900000,131903.0
25022,2025-2026,1642962.0,Drake,Powell,Regular Season,6.200000,1.500000,1.680000,0.160000,0.500000,...,75.0,84.0,8.0,25.0,50.0,0.525490,0.419608,0.291045,0.904762,3372240.0
25023,2025-2026,1642964.0,Brooks,Barnhizer,Regular Season,1.424242,0.545455,1.727273,0.121212,0.181818,...,18.0,57.0,4.0,6.0,8.0,0.340000,0.380000,0.294118,0.666667,636435.0


In [6]:
# stats to normalize and rank within each season
stat_cols = [
    "points", "assists", "rebounds", "blocks", "steals", "turnovers",
    "fieldGoalsMade", "fieldGoalsAttempted", "threePointersMade", "threePointersAttempted",
    "freeThrowsMade", "freeThrowsAttempted", "fieldGoalsPercentage", "threePointersPercentage", 
    "freeThrowsPercentage", "proportionThreePoint", "numMinutes", "win", "gamesPlayed", "salary", 
    "plusMinusPoints", "foulsPersonal",
]

# Metrics where lower is better (rank ascending)
lower_is_better = {"turnovers", "foulsPersonal"}
ranked_stats = player_stats_season_flat.copy()

for col in stat_cols:
    values = ranked_stats[col]
    season_min = ranked_stats.groupby(["season", "gameType"])[col].transform("min")
    season_max = ranked_stats.groupby(["season", "gameType"])[col].transform("max")
    denom = (season_max - season_min).replace(0, pd.NA)

    # Min-max normalization per season.
    # Keep missing source values as NA (e.g. salary not available in early seasons).
    norm = (values - season_min) / denom
    norm = norm.where(values.notna(), pd.NA)
    norm = norm.mask(values.notna() & norm.isna(), 0.0)
    ranked_stats[f"{col}_norm"] = norm

    # Rank per season (1 = best), preserving missing values as <NA>
    ranks = ranked_stats.groupby(["season", "gameType"])[col].rank(
        method="dense",
        ascending=(col in lower_is_better),
        na_option="keep"
    )
    ranked_stats[f"{col}_rank"] = ranks.astype("Int64")

# save in csv format
ranked_stats.to_csv(DATA_PATH + "/processed/player_seasons.csv", index=False)

display(ranked_stats)

,season,personId,firstName,lastName,gameType,points,assists,rebounds,blocks,steals,...,win_norm,win_rank,gamesPlayed_norm,gamesPlayed_rank,salary_norm,salary_rank,plusMinusPoints_norm,plusMinusPoints_rank,foulsPersonal_norm,foulsPersonal_rank
0,1980-1981,229.0,James,Edwards,Regular Season,15.629630,2.617284,7.049383,1.382716,0.098765,...,0.709677,18,0.956522,3,NaN,<NA>,0.0,1,0.888109,247
1,1980-1981,305.0,Robert,Parish,Playoffs,15.000000,1.117647,8.588235,2.117647,0.411765,...,1.000000,1,0.764706,4,NaN,<NA>,0.0,1,0.980944,66
2,1980-1981,305.0,Robert,Parish,Regular Season,18.926829,1.768293,9.475610,1.939024,0.402439,...,1.000000,1,0.971014,2,NaN,<NA>,0.0,1,0.895683,250
3,1980-1981,328.0,Rick,Mahorn,Regular Season,4.788462,0.480769,4.134615,0.480769,0.057692,...,0.387097,38,0.536232,31,NaN,<NA>,0.0,1,0.563013,149
4,1980-1981,940.0,Earl,Cureton,Playoffs,1.333333,0.222222,1.000000,0.222222,0.111111,...,0.454545,7,0.294118,10,NaN,<NA>,0.0,1,0.075117,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25020,2025-2026,1642959.0,Chris,Youngblood,Regular Season,2.031250,0.343750,0.875000,0.093750,0.125000,...,0.531915,23,0.326923,36,0.009559,306,0.468617,186,0.136004,20
25021,2025-2026,1642959.0,Chris,Youngblood,Regular Season,2.031250,0.343750,0.875000,0.093750,0.125000,...,0.531915,23,0.326923,36,0.001086,323,0.468617,186,0.136004,20
25022,2025-2026,1642962.0,Drake,Powell,Regular Season,6.200000,1.500000,1.680000,0.160000,0.500000,...,0.234043,37,0.673077,18,0.055509,233,0.158574,411,0.351713,120
25023,2025-2026,1642964.0,Brooks,Barnhizer,Regular Season,1.424242,0.545455,1.727273,0.121212,0.181818,...,0.468085,26,0.346154,35,0.009559,306,0.372704,297,0.139167,21


## NBA Basketball court

In [7]:
shotdetails = pd.read_csv(DATA_PATH + "/nba_play_by_play_shot_data/shotdetail_2015.csv")
shotdetails

,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM
0,Shot Chart Detail,21500001,6,200794,Paul Millsap,1610612737,Atlanta Hawks,1,11,0,...,Right Side(R),8-16 ft.,12,76,95,1,1,20151027,ATL,DET
1,Shot Chart Detail,21500001,8,201143,Al Horford,1610612737,Atlanta Hawks,1,10,27,...,Left Side Center(LC),16-24 ft.,20,-117,164,1,0,20151027,ATL,DET
2,Shot Chart Detail,21500001,13,200794,Paul Millsap,1610612737,Atlanta Hawks,1,10,1,...,Right Side(R),8-16 ft.,12,123,3,1,1,20151027,ATL,DET
3,Shot Chart Detail,21500001,20,201952,Jeff Teague,1610612737,Atlanta Hawks,1,9,29,...,Center(C),8-16 ft.,15,-2,154,1,0,20151027,ATL,DET
4,Shot Chart Detail,21500001,23,201952,Jeff Teague,1610612737,Atlanta Hawks,1,9,12,...,Center(C),Less Than 8 ft.,1,12,2,1,0,20151027,ATL,DET
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207888,Shot Chart Detail,21501221,557,1626162,Kelly Oubre Jr.,1610612764,Washington Wizards,4,3,50,...,Left Side(L),8-16 ft.,8,-86,21,1,0,20160413,WAS,ATL
207889,Shot Chart Detail,21501221,561,201196,Ramon Sessions,1610612764,Washington Wizards,4,3,31,...,Right Side Center(RC),24+ ft.,25,96,237,1,0,20160413,WAS,ATL
207890,Shot Chart Detail,21501221,563,204067,Jarell Eddie,1610612764,Washington Wizards,4,3,22,...,Center(C),Less Than 8 ft.,0,-4,7,1,0,20160413,WAS,ATL
207891,Shot Chart Detail,21501221,587,201162,Jared Dudley,1610612764,Washington Wizards,4,1,49,...,Left Side Center(LC),16-24 ft.,17,-104,139,1,1,20160413,WAS,ATL
